# RAG System Evaluation

This notebook evaluates the RAG Q&A system across 4 key metrics:

| Metric | What it measures |
|--------|------------------|
| **Context Precision** | Are the retrieved chunks actually relevant to the question? |
| **Faithfulness** | Does the answer only use information from the retrieved context? |
| **Answer Relevancy** | Does the answer actually address what was asked? |
| **Gap Detection Accuracy** | Does the system correctly flag out-of-domain questions? |

We compute these manually so we understand every number in the results.

In [1]:
import sys
sys.path.append('../backend')

import pandas as pd
from retriever import retrieve
from chain import ask

C:\Users\NEHAL\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13114.76it/s]


## Step 1: Define the Evaluation Dataset

We need 3 types of questions:
- **In-domain**: questions the document clearly answers (expect high confidence)
- **Partial**: questions the document partially covers (expect medium confidence)
- **Out-of-domain**: questions the document doesn't cover (expect knowledge gap flag)

Each in-domain question has a `ground_truth` — the correct answer we compare against.

In [2]:
eval_dataset = [
    # In-domain questions — document should answer these well
    {
        'question': 'What is the main problem this paper addresses?',
        'ground_truth': 'Microplastic pollution in riverine ecosystems and inadequate modeling of spatial heterogeneity and topological complexity',
        'type': 'in-domain'
    },
    {
        'question': 'What does the spatiotemporal graph convolutional network do?',
        'ground_truth': 'It identifies microplastic migration pathways and traces pollution sources by integrating spatial dependency learning with temporal sequence modeling',
        'type': 'in-domain'
    },
    {
        'question': 'How does the model handle spatial and temporal data?',
        'ground_truth': 'It uses stacked layers alternating between spatial graph convolution modules and temporal convolution blocks',
        'type': 'in-domain'
    },
    {
        'question': 'What type of pollution does this paper focus on?',
        'ground_truth': 'Microplastic pollution in riverine ecosystems',
        'type': 'in-domain'
    },
    {
        'question': 'What is the graph structure used in the model?',
        'ground_truth': 'Multi-scale graph representations encoding system structure and transport dynamics',
        'type': 'in-domain'
    },
    # Out-of-domain questions — should trigger knowledge gap
    {
        'question': 'What is the recipe for chocolate cake?',
        'ground_truth': None,
        'type': 'out-of-domain'
    },
    {
        'question': 'Who won the FIFA World Cup in 2022?',
        'ground_truth': None,
        'type': 'out-of-domain'
    },
    {
        'question': 'How do I write a Python web scraper?',
        'ground_truth': None,
        'type': 'out-of-domain'
    },
]

print(f'Evaluation dataset: {len(eval_dataset)} questions')
print(f'  In-domain:     {sum(1 for q in eval_dataset if q["type"] == "in-domain")}')
print(f'  Out-of-domain: {sum(1 for q in eval_dataset if q["type"] == "out-of-domain")}')

Evaluation dataset: 8 questions
  In-domain:     5
  Out-of-domain: 3


## Step 2: Run the RAG System on Every Question

We call our actual system for each question and collect the answers, retrieved contexts, confidence scores, and gap flags.

In [3]:
results = []

for item in eval_dataset:
    print(f'Running: {item["question"][:60]}...')
    output = ask(item['question'])
    
    results.append({
        'question':        item['question'],
        'type':            item['type'],
        'ground_truth':    item['ground_truth'],
        'answer':          output['answer'],
        'confidence_level': output['confidence']['level'],
        'confidence_score': output['confidence']['score'],
        'is_knowledge_gap': output['confidence']['is_knowledge_gap'],
        'top_chunk_score':  output['sources'][0]['score'] if output['sources'] else 0,
        'contexts':        [s for s in output['sources']]
    })

print('\nDone.')

Running: What is the main problem this paper addresses?...
Running: What does the spatiotemporal graph convolutional network do?...
Running: How does the model handle spatial and temporal data?...
Running: What type of pollution does this paper focus on?...
Running: What is the graph structure used in the model?...
Running: What is the recipe for chocolate cake?...
Running: Who won the FIFA World Cup in 2022?...
Running: How do I write a Python web scraper?...

Done.


## Step 3: Compute Metrics

### Metric 1: Context Precision
For each in-domain question, we check if the top retrieved chunk score is above the medium confidence threshold (0.35). A score above this means the retrieval found genuinely relevant content.

In [4]:
MEDIUM_THRESHOLD = 0.35

in_domain_results = [r for r in results if r['type'] == 'in-domain']

context_precision_scores = [
    1.0 if r['top_chunk_score'] >= MEDIUM_THRESHOLD else 0.0
    for r in in_domain_results
]

context_precision = sum(context_precision_scores) / len(context_precision_scores)
print(f'Context Precision: {context_precision:.2f} ({sum(context_precision_scores):.0f}/{len(context_precision_scores)} questions retrieved relevant context)')

Context Precision: 0.80 (4/5 questions retrieved relevant context)


### Metric 2: Faithfulness
We check whether the answer contains phrases that indicate the model is answering from context vs hallucinating. Hallucination indicators: "I think", "I believe", "probably", "I'm not sure", "I don't know" — these suggest the model is guessing rather than citing the document.

In [5]:
hallucination_phrases = [
    "i think", "i believe", "probably", "i'm not sure",
    "i don't know", "i cannot", "i am not certain", "might be"
]

faithfulness_scores = []
for r in in_domain_results:
    answer_lower = r['answer'].lower()
    has_hallucination = any(phrase in answer_lower for phrase in hallucination_phrases)
    faithfulness_scores.append(0.0 if has_hallucination else 1.0)

faithfulness = sum(faithfulness_scores) / len(faithfulness_scores)
print(f'Faithfulness: {faithfulness:.2f} ({sum(faithfulness_scores):.0f}/{len(faithfulness_scores)} answers appear grounded in context)')

Faithfulness: 1.00 (5/5 answers appear grounded in context)


### Metric 3: Answer Relevancy
We measure word overlap between the generated answer and the ground truth. This is a simple but interpretable proxy — a high overlap means the answer covers the right concepts.

In [6]:
def word_overlap_score(answer: str, ground_truth: str) -> float:
    """Proportion of ground truth keywords found in the answer."""
    # Filter out common stop words
    stop_words = {'the', 'a', 'an', 'is', 'it', 'in', 'of', 'to', 'and', 'or', 'for', 'with', 'by', 'this', 'that'}
    
    truth_words = set(ground_truth.lower().split()) - stop_words
    answer_words = set(answer.lower().split()) - stop_words
    
    if not truth_words:
        return 0.0
    
    return len(truth_words & answer_words) / len(truth_words)

relevancy_scores = [
    word_overlap_score(r['answer'], r['ground_truth'])
    for r in in_domain_results
]

answer_relevancy = sum(relevancy_scores) / len(relevancy_scores)
print(f'Answer Relevancy: {answer_relevancy:.2f} (average keyword overlap with ground truth)')

Answer Relevancy: 0.41 (average keyword overlap with ground truth)


### Metric 4: Gap Detection Accuracy
How accurately does our system flag out-of-domain questions? This is the metric unique to our project.

In [7]:
out_of_domain_results = [r for r in results if r['type'] == 'out-of-domain']

# Correctly flagged = out-of-domain question AND is_knowledge_gap is True
correctly_flagged = sum(1 for r in out_of_domain_results if r['is_knowledge_gap'])
gap_detection_accuracy = correctly_flagged / len(out_of_domain_results)

# Also check in-domain questions are NOT flagged as gaps (false positives)
false_positives = sum(1 for r in in_domain_results if r['is_knowledge_gap'])

print(f'Gap Detection Accuracy: {gap_detection_accuracy:.2f} ({correctly_flagged}/{len(out_of_domain_results)} out-of-domain questions correctly flagged)')
print(f'False Positives: {false_positives}/{len(in_domain_results)} in-domain questions incorrectly flagged as gaps')

Gap Detection Accuracy: 1.00 (3/3 out-of-domain questions correctly flagged)
False Positives: 0/5 in-domain questions incorrectly flagged as gaps


## Step 4: Results Summary

In [8]:
summary = pd.DataFrame({
    'Metric': ['Context Precision', 'Faithfulness', 'Answer Relevancy', 'Gap Detection Accuracy'],
    'Score': [context_precision, faithfulness, answer_relevancy, gap_detection_accuracy],
    'Description': [
        'Retrieved chunks were relevant to the question',
        'Answers grounded in context, no hallucination indicators',
        'Answer keyword overlap with ground truth',
        'Out-of-domain questions correctly flagged'
    ]
})

summary['Score'] = summary['Score'].map('{:.2f}'.format)
print(summary.to_string(index=False))

                Metric Score                                              Description
     Context Precision  0.80           Retrieved chunks were relevant to the question
          Faithfulness  1.00 Answers grounded in context, no hallucination indicators
      Answer Relevancy  0.41                 Answer keyword overlap with ground truth
Gap Detection Accuracy  1.00                Out-of-domain questions correctly flagged


## Step 5: Per-Question Breakdown

In [9]:
rows = []
for r in results:
    rows.append({
        'Question': r['question'][:55] + '...',
        'Type': r['type'],
        'Confidence': r['confidence_level'],
        'Score': round(r['confidence_score'], 3),
        'Gap Flagged': r['is_knowledge_gap'],
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

                                                  Question          Type Confidence  Score  Gap Flagged
         What is the main problem this paper addresses?...     in-domain        low  0.251        False
What does the spatiotemporal graph convolutional networ...     in-domain       high  0.615        False
   How does the model handle spatial and temporal data?...     in-domain     medium  0.434        False
       What type of pollution does this paper focus on?...     in-domain     medium  0.454        False
         What is the graph structure used in the model?...     in-domain     medium  0.546        False
                 What is the recipe for chocolate cake?... out-of-domain        low  0.141         True
                    Who won the FIFA World Cup in 2022?... out-of-domain        low  0.124         True
                   How do I write a Python web scraper?... out-of-domain        low  0.178         True
